# Detector evaluation - dual track, GPT-5.6 Luna items

Six detectors judge each (source, summary) pair as hallucinated or not. The same six used for the NER track.

The detector sees **only a source document and the summary**. It is never told which source it got, what was edited, or that anything was edited.

| Track | Pair | Gold |
|---|---|---|
| A | original source + summary | 0, faithful |
| B | corrupted source + summary | 1, hallucinated |

Track A is not optional. Scored on corrupted items alone, a model that answers "hallucinated" every time scores 100%.

**Internal reasoning is ENABLED.** The `reasoning` field is omitted entirely, so each model uses its own default; the three that reason will reason. Because reasoning tokens count against the output budget, `max_tokens` is 2048 rather than 64 - at 64 a reasoning model spends the whole budget thinking and returns nothing, which is exactly what truncated DeepSeek during generation.

Reasoning token counts are recorded per call so the cost is visible afterwards.

Output is one CSV per detector in `evaluation_summarization/`, matching the NER track's layout.

In [1]:
import os
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY_NEW"), os.getenv("OPENROUTER_API_KEY")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

DETECTORS = {
    "gpt_4o_mini":           "openai/gpt-4o-mini",
    "deepseek_v4_flash":     "deepseek/deepseek-v4-flash",
    "gemini_3_1_flash_lite": "google/gemini-3.1-flash-lite",
    "llama_3_1_70b":         "meta-llama/llama-3.1-70b-instruct",
    "qwen_2_5_72b":          "qwen/qwen-2.5-72b-instruct",
    "grok_4_3":              "x-ai/grok-4.3",
}

GENERATOR = "deepseek"
OUT_DIR = "../../evaluation_summarization"
BACKUP_FILE = f"backup_detector_verdicts_{GENERATOR}.csv"
CHECKPOINT_EVERY = 50
MAX_WORKERS = 6
MAX_TOKENS = 2048

os.makedirs(OUT_DIR, exist_ok=True)
print(f"{len(api_keys)} key(s). {len(DETECTORS)} detectors on '{GENERATOR}' items.")
print(f"Output dir: {os.path.abspath(OUT_DIR)}")

2 key(s). 6 detectors on 'deepseek' items.
Output dir: C:\Users\ACER\Desktop\benmedhallu-local\evaluation_summarization


In [2]:
items = []

for _, row in pd.read_csv("eval_track_a.csv").iterrows():
    items.append({"item_id": f"A_{row['doc_id']}", "track": "A", "gold": 0,
                  "method": "", "n": 0, "target_categories": "",
                  "doc_id": row["doc_id"], "source": row["source_original"],
                  "summary": row["summary"]})

for i, row in pd.read_csv(f"eval_{GENERATOR}.csv").iterrows():
    items.append({"item_id": f"B_{i}", "track": "B", "gold": 1,
                  "method": row["method"], "n": row["n"],
                  "target_categories": row["target_categories"],
                  "doc_id": row["doc_id"], "source": row["source_corrupted"],
                  "summary": row["summary"]})

items = pd.DataFrame(items)
assert items["item_id"].is_unique, "duplicate item_id"
assert items["source"].notna().all() and items["summary"].notna().all(), "blank source or summary"

print(f"{len(items)} items: {(items['track'] == 'A').sum()} Track A, "
      f"{(items['track'] == 'B').sum()} Track B")
print(f"{len(items) * len(DETECTORS)} detector calls")

250 items: 50 Track A, 200 Track B
1500 detector calls


In [3]:
DETECT_PROMPT = """You are a medical verification assistant.

Determine whether the summary contains information that is not supported by, or that contradicts, the source document.

Judge only against the source document. Do not use outside medical knowledge. A summary that is incomplete but contains nothing unsupported is NOT hallucinated.

Source document:
{source}

Summary:
{summary}

If the summary contains hallucinated information, output:
ANSWER: 1

Otherwise, output:
ANSWER: 0
"""

key_lock = threading.Lock()
active_key_index = 0


def call_model(prompt, model_name, max_retries=3):
    """No `reasoning` field: each model uses its own default, so the three that
    reason will reason. MAX_TOKENS must stay well above the reasoning budget or
    those models spend it all thinking and return an empty answer."""
    global active_key_index
    payload = {"model": model_name, "messages": [{"role": "user", "content": prompt}],
               "temperature": 0.0, "max_tokens": MAX_TOKENS}
    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            with key_lock:
                k = api_keys[active_key_index]
            try:
                r = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {k}", "Content-Type": "application/json"},
                    json=payload, timeout=180)
            except requests.RequestException:
                break
            if r.status_code == 200:
                body = r.json()
                usage = body.get("usage", {}) or {}
                detail = usage.get("completion_tokens_details", {}) or {}
                return (body["choices"][0]["message"]["content"],
                        detail.get("reasoning_tokens", 0),
                        usage.get("cost", 0.0),
                        body["choices"][0].get("finish_reason"))
            if r.status_code in (401, 402, 403, 429):
                with key_lock:
                    active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            break
        time.sleep(2 * (attempt + 1))
    return None, 0, 0.0, "error"


def parse_verdict(text):
    """1, 0, or None. None means no usable verdict came back."""
    if not text:
        return None
    m = re.search(r"ANSWER:\s*([01])", text)
    if m:
        return int(m.group(1))
    m = re.search(r"\b([01])\b", text.strip())
    return int(m.group(1)) if m else None


def run_one(task):
    item, name, slug = task
    raw, reasoning_tokens, cost, finish = call_model(
        DETECT_PROMPT.format(source=item["source"], summary=item["summary"]), slug)
    verdict = parse_verdict(raw)
    return {"job_key": f"{name}|{item['item_id']}", "detector": name,
            "item_id": item["item_id"], "track": item["track"], "gold": item["gold"],
            "doc_id": item["doc_id"], "method": item["method"], "n": item["n"],
            "target_categories": item["target_categories"],
            "prediction": verdict,
            "correct": None if verdict is None else int(verdict == item["gold"]),
            "reasoning_tokens": reasoning_tokens, "cost": cost,
            "finish_reason": finish, "raw_response": raw}


tasks = [(row, name, slug) for _, row in items.iterrows() for name, slug in DETECTORS.items()]
assert len({f"{n}|{r['item_id']}" for r, n, _ in tasks}) == len(tasks), "duplicate job_key"
print(f"{len(tasks)} calls queued.")

1500 calls queued.


In [4]:
# PRE-FLIGHT. One call per detector before committing to the rest.
# With reasoning on, the thing to watch is finish_reason: 'length' means the
# model burned MAX_TOKENS thinking and never answered.
probe = items.iloc[0]
smoke_ok = True
print(f"{'':>6}  {'detector':<24} {'verdict':>7} {'reason_tok':>10} {'finish':>10}  cost")
for name, slug in DETECTORS.items():
    raw, rt, cost, finish = call_model(
        DETECT_PROMPT.format(source=probe["source"], summary=probe["summary"]), slug)
    verdict = parse_verdict(raw)
    if verdict is None:
        smoke_ok = False
    print(f"{'OK' if verdict is not None else 'FAILED':>6}  {name:<24} "
          f"{str(verdict):>7} {rt:>10} {str(finish):>10}  ${cost:.5f}")

if not smoke_ok:
    raise RuntimeError("Pre-flight failed. Do not run the full evaluation.")
print("")
print("Pre-flight passed.")

        detector                 verdict reason_tok     finish  cost


    OK  gpt_4o_mini                    0          0       stop  $0.00004


    OK  deepseek_v4_flash              0        991       stop  $0.00015


    OK  gemini_3_1_flash_lite          0          0       stop  $0.00007


    OK  llama_3_1_70b                  0          0       stop  $0.00047


    OK  qwen_2_5_72b                   0          0       stop  $0.00023


    OK  grok_4_3                       0        782       stop  $0.00240

Pre-flight passed.


In [5]:
done = {}
if os.path.exists(BACKUP_FILE):
    prior = pd.read_csv(BACKUP_FILE)
    done = {r["job_key"]: r for _, r in prior.iterrows()}
    print(f"Resuming: {len(done)} verdicts already collected.")
else:
    print("No backup - starting fresh.")

pending = [t for t in tasks if f"{t[1]}|{t[0]['item_id']}" not in done]
print(f"{len(pending)} of {len(tasks)} calls still to make.")
print("")

start = time.time()
results = [done[f"{n}|{r['item_id']}"].to_dict()
           for r, n, _ in tasks if f"{n}|{r['item_id']}" in done]
lock = threading.Lock()


def checkpoint():
    pd.DataFrame(results).to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")


if pending:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for finished, record in enumerate(pool.map(run_one, pending), start=1):
            with lock:
                results.append(record)
                if finished % CHECKPOINT_EVERY == 0 or finished == len(pending):
                    checkpoint()
                    usable = sum(r.get("prediction") is not None for r in results)
                    spent = sum(float(r.get("cost") or 0) for r in results)
                    print(f"   --- saved at {finished}/{len(pending)} "
                          f"(usable={usable}, ${spent:.3f}, {time.time() - start:.0f}s) ---")

checkpoint()
v = pd.DataFrame(results)
print("")
print(f"{len(v)} verdicts, {int(v['prediction'].notna().sum())} usable, "
      f"total cost ${v['cost'].astype(float).sum():.3f}")

No backup - starting fresh.
1500 of 1500 calls still to make.



   --- saved at 50/1500 (usable=50, $0.021, 32s) ---


   --- saved at 100/1500 (usable=100, $0.045, 60s) ---


   --- saved at 150/1500 (usable=150, $0.070, 85s) ---


   --- saved at 200/1500 (usable=200, $0.093, 111s) ---


...

   --- saved at 1300/1500 (usable=1296, $0.595, 696s) ---


   --- saved at 1350/1500 (usable=1346, $0.613, 717s) ---


   --- saved at 1400/1500 (usable=1396, $0.640, 751s) ---


   --- saved at 1450/1500 (usable=1446, $0.662, 778s) ---


   --- saved at 1500/1500 (usable=1495, $0.683, 805s) ---

1500 verdicts, 1495 usable, total cost $0.683


In [6]:
# One CSV per detector, matching the NER track's layout.
COLS = ["doc_id", "item_id", "track", "gold", "method", "n", "target_categories",
        "prediction", "correct", "reasoning_tokens", "finish_reason", "raw_response"]

print(f"{'file':<62} {'rows':>5} {'acc%':>6}")
print("-" * 76)
for name in DETECTORS:
    d = v[v["detector"] == name][COLS]
    path = os.path.join(OUT_DIR, f"{GENERATOR}_summarization_evaluated_by_{name}.csv")
    d.to_csv(path, index=False, encoding="utf-8-sig")
    acc = 100 * d["correct"].dropna().mean()
    print(f"{os.path.basename(path):<62} {len(d):>5} {acc:>6.1f}")

file                                                            rows   acc%
----------------------------------------------------------------------------
deepseek_summarization_evaluated_by_gpt_4o_mini.csv              250   74.4
deepseek_summarization_evaluated_by_deepseek_v4_flash.csv        250   89.1
deepseek_summarization_evaluated_by_gemini_3_1_flash_lite.csv    250   91.6
deepseek_summarization_evaluated_by_llama_3_1_70b.csv            250   56.7
deepseek_summarization_evaluated_by_qwen_2_5_72b.csv             250   85.2
deepseek_summarization_evaluated_by_grok_4_3.csv                 250   92.4


In [7]:
scored = v.dropna(subset=["prediction"]).copy()
scored["wrong"] = scored["correct"].astype(int) == 0

rows = []
for name in DETECTORS:
    d = scored[scored["detector"] == name]
    a = 100 * d[d["track"] == "A"]["wrong"].mean()
    b = 100 * d[d["track"] == "B"]["wrong"].mean()
    rows.append({"detector": name, "accuracy": round(100 * d["correct"].mean(), 1),
                 "A_err_false_alarm": round(a, 1), "B_err_missed": round(b, 1),
                 "balanced_err": round((a + b) / 2, 1),
                 "reason_tok": int(d["reasoning_tokens"].astype(float).mean()),
                 "cost": round(d["cost"].astype(float).sum(), 3)})
summary = pd.DataFrame(rows).sort_values("balanced_err")
summary.to_csv(os.path.join(OUT_DIR, f"{GENERATOR}_summarization_scores.csv"),
               index=False, encoding="utf-8-sig")
print("Balanced error rate: lower is better, a uniform responder scores 50.")
print("")
print(summary.to_string(index=False))

b = scored[scored["track"] == "B"].copy()
b["variant"] = b.apply(
    lambda r: r["method"] if r["method"] != "Delete" else f"Delete N={int(r['n'])}", axis=1)
print("")
print("MISS RATE BY VARIANT (% of hallucinated items called faithful)")
print("")
print((100 * b.pivot_table(index="variant", columns="detector", values="wrong")).round(0).to_string())

print("")
print("MISS RATE BY FIRST TARGET CATEGORY")
print("")
b["cat"] = b["target_categories"].astype(str).str.split(" | ", regex=False).str[0]
pivot = 100 * b.pivot_table(index="cat", columns="detector", values="wrong")
pivot["items"] = b.groupby("cat").size() / len(DETECTORS)
print(pivot.round(0).to_string())

Balanced error rate: lower is better, a uniform responder scores 50.

             detector  accuracy  A_err_false_alarm  B_err_missed  balanced_err  reason_tok  cost
             grok_4_3      92.4                8.0           7.5           7.8         582 0.493
gemini_3_1_flash_lite      91.6               10.0           8.0           9.0           0 0.015
    deepseek_v4_flash      89.1                6.0          12.1           9.1         252 0.019
         qwen_2_5_72b      85.2               10.0          16.0          13.0           0 0.062
          gpt_4o_mini      74.4                6.0          30.5          18.2           0 0.011
        llama_3_1_70b      56.7                4.0          53.3          28.6           0 0.081

MISS RATE BY VARIANT (% of hallucinated items called faithful)

detector    deepseek_v4_flash  gemini_3_1_flash_lite  gpt_4o_mini  grok_4_3  llama_3_1_70b  qwen_2_5_72b
variant                                                                          